In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install duckdb==1.4.4 polars==1.40.0 --quiet

In [ ]:
# ── Cell 2: Mount Drive and copy DB to local ──────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil

DRIVE_DB  = Path("/content/drive/MyDrive/b3_data/db/b3_data.duckdb")
LOCAL_DB  = Path("/content/b3_data.duckdb")

if not DRIVE_DB.exists():
    raise FileNotFoundError(
        f"Database not found at {DRIVE_DB}.\n"
        "Run colab_ingest.ipynb first to build the database."
    )

print(f"Copying DB from Drive → {LOCAL_DB} ...")
shutil.copy2(DRIVE_DB, LOCAL_DB)
size_mb = LOCAL_DB.stat().st_size / 1_048_576
print(f"Ready. DB size: {size_mb:.1f} MB")

In [ ]:
# ── Cell 3: Open connection and set resource limits ────────────────────────
import duckdb

con = duckdb.connect(str(LOCAL_DB))

# Conservative limits suitable for Colab free tier (12 GB RAM, 2 vCPUs)
con.execute("SET threads TO 2")
con.execute("SET memory_limit = '4GB'")

print("Connection open. DuckDB version:", duckdb.__version__)

In [ ]:
# ── Example 1: Daily close prices for a specific ticker ────────────────────
import plotly.express as px

TICKER = "PETR4"   # Change this to any ticker in your dataset
YEAR   = 2025      # Change to any year in your dataset

df_prices = con.sql(f"""
    SELECT datpre, preult AS close, voltot AS volume_R$
    FROM cotacoes
    WHERE codneg = '{TICKER}'
      AND YEAR(datpre) = {YEAR}
      AND codbdi = '02'          -- ordinary shares, spot market
    ORDER BY datpre
""").df()

if len(df_prices) > 0:
    fig = px.line(
        df_prices,
        x="datpre",
        y="close",
        title=f"{TICKER} — Daily Close Price ({YEAR})",
        labels={"datpre": "Date", "close": "Close (R$)"},
    )
    fig.show()
else:
    print(f"No data found for {TICKER} in {YEAR}")

In [ ]:
# ── Example 2: Top 10 tickers by total traded volume ──────────────────────
YEAR = 2025  # Change to any year in your dataset

df_volume = con.sql(f"""
    SELECT
        codneg,
        nomres,
        SUM(voltot)  AS volume_total_R$,
        SUM(totneg)  AS total_trades,
        SUM(quatot)  AS total_shares
    FROM cotacoes
    WHERE YEAR(datpre) = {YEAR}
      AND codbdi = '02'
    GROUP BY codneg, nomres
    ORDER BY volume_total_R$ DESC
    LIMIT 10
""").df()

fig = px.bar(
    df_volume,
    x="codneg",
    y="volume_total_R$",
    title=f"Top 10 Tickers by Volume — {YEAR}",
    labels={"codneg": "Ticker", "volume_total_R$": "Total Volume (R$)"},
    text_auto=".2s",
)
fig.show()
display(df_volume)

In [ ]:
# ── Example 3: Options chain for a specific underlying ────────────────────
# tpmerc = '070' → call options; '080' → put options (B3 market type codes)
UNDERLYING = "PETR"   # codneg usually starts with underlying ticker root
SNAPSHOT_DATE = "2025-06-20"   # expiry date to filter

df_options = con.sql(f"""
    SELECT
        codneg,
        especi,
        tpmerc,
        datven    AS expiry,
        preexe    AS strike,
        preult    AS last_price,
        voltot    AS volume_R$,
        totneg    AS trades
    FROM cotacoes
    WHERE codneg LIKE '{UNDERLYING}%'
      AND tpmerc IN ('070', '080')
      AND datven = '{SNAPSHOT_DATE}'
    ORDER BY tpmerc, strike
""").df()

if len(df_options) > 0:
    display(df_options)
else:
    print(f"No options data found for {UNDERLYING} expiring {SNAPSHOT_DATE}")

In [ ]:
# ── Options: Best profit opportunities — parameters ────────────────────────
START_DATE = "2025-01-01"   # Start of analysis window
END_DATE   = "2025-12-31"   # End of analysis window
TOP_N      = 20             # How many results to show
MIN_TRADES = 10             # Minimum total trades (filters illiquid options)

In [ ]:
# ── Options: Best profit opportunities ────────────────────────────────────
# Profit % = (best sell price) - (best buy price) / (best buy price) × 100
# Buy and sell prices are guaranteed to be in chronological order:
# running_min_buy is the cheapest premin seen on any day *before* the sell day.

df_opts = con.sql(f"""
    WITH daily AS (
        SELECT codneg, tpmerc, nomres, datven, datpre, premin, premax, totneg
        FROM cotacoes
        WHERE tpmerc IN ('070', '080')
          AND premin > 0
          AND datpre BETWEEN '{START_DATE}' AND '{END_DATE}'
    ),
    running_min AS (
        SELECT
            codneg, tpmerc, nomres, datven, datpre, premax, totneg,
            MIN(premin) OVER (
                PARTITION BY codneg
                ORDER BY datpre
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ) AS running_min_buy
        FROM daily
    ),
    profit_per_day AS (
        SELECT
            codneg, tpmerc, nomres, datven,
            running_min_buy                                               AS min_buy_price,
            premax                                                        AS max_sell_price,
            ROUND((premax - running_min_buy) / running_min_buy * 100, 2) AS profit_pct
        FROM running_min
        WHERE running_min_buy IS NOT NULL
          AND premax > running_min_buy
    ),
    trade_counts AS (
        SELECT codneg, SUM(totneg) AS total_trades
        FROM daily
        GROUP BY codneg
    )
    SELECT
        p.codneg,
        CASE p.tpmerc WHEN '070' THEN 'CALL' WHEN '080' THEN 'PUT' END AS type,
        p.nomres,
        p.datven        AS expiry_date,
        p.min_buy_price,
        p.max_sell_price,
        p.profit_pct,
        t.total_trades
    FROM profit_per_day p
    JOIN trade_counts t ON p.codneg = t.codneg
    WHERE t.total_trades >= {MIN_TRADES}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY p.codneg ORDER BY p.profit_pct DESC) = 1
    ORDER BY p.profit_pct DESC
    LIMIT {TOP_N}
""").df()

if len(df_opts) > 0:
    df_opts.index = range(1, len(df_opts) + 1)
    display(df_opts.style.format({
        "min_buy_price":  "R$ {:.2f}",
        "max_sell_price": "R$ {:.2f}",
        "profit_pct":     "{:.2f}%",
    }).bar(subset=["profit_pct"], color="#4caf50"))
else:
    print(f"No options data found between {START_DATE} and {END_DATE}")

In [ ]:
# ── Cell 7: Save: sync DB back to Drive ────────────────────────────────────
# Run this before closing the session if you have written any data
# (e.g., created views or additional tables).
con.close()

shutil.copy2(LOCAL_DB, DRIVE_DB)
print(f"DB synced to Drive. Size: {DRIVE_DB.stat().st_size / 1_048_576:.1f} MB")